# L3d · Recuperación de punta a punta con tus documentos (zonas `test` y `val`)

Un solo notebook, limpio, que va de los PDF a los candidatos que recibiría el juez:

1. **Secciones** de la norma y del manual (Docling real, seccionado, exclusión de lo no comparable) en una **tabla**.
2. **Chunks:** cuántos sub-chunks y de cuántos tokens sale cada sección (la tabla del paso 1 ya los cuenta).
3. **Embeddings + FAISS** con el modelo del stage (`config/stages/<STAGE>.env`) y, si lo activas, **reranker** (`RERANK_ENABLED`).
4. **Recuperación:** los pares (artículo, sección) que se juzgarían, con ruta, páginas, score y rango de cada vía, y diagnósticos (artículos sin buenos candidatos, secciones del manual que nadie recupera).
5. **Validación opcional** contra tu comparación hecha a mano (recall con intervalo de confianza).

## Zonas de documentos
| Zona | Qué lleva | Carpetas | Git |
|---|---|---|---|
| `test` | MOCK ficticios y normas públicas | `documentos/test/{normas,manuales,comparaciones}`, `output/test/` | los MOCK y los resultados de `output/test/` se versionan (no la caché de bloques) |
| `val` | **documentos reales** de validación | `documentos/val/{normas,manuales,comparaciones}`, `output/val/` | **nunca**: todo `val` está ignorado |

Cambia `ZONA` en la celda 2. La caché de Docling (`output/<zona>/bloques/`) lleva el texto íntegro y tampoco se versiona nunca.

**Confidencialidad en `val`:** el notebook no muestra texto de los documentos (solo identificadores, títulos, rutas, páginas, scores). Aun así, antes de hacer commit del notebook: *Restart & Clear All Outputs*.

In [ ]:
# --- Arranque por stage (ver 00_modelos.ipynb) ---------------------------------------------------------------------------
# Local: el stage sale de config/.env (dev: Mac + DMR; sandbox: Raspberry). Databricks (mvp): ver la celda de arranque de los otros notebooks.

In [ ]:
import os, sys, re
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ)); os.chdir(RAIZ)

import numpy as np
import pandas as pd
from IPython.display import display
from backend.config import settings as s, carpeta_bloques, carpeta_comparaciones, carpeta_manuales, carpeta_normas, carpeta_salida
from backend.core.progreso import configurar_logs, progreso
from backend.engine.pares import construir_pares
from backend.extraction.lector import leer_documento
from backend.llm.client import ModelClient
from backend.retrieval import seleccion
from backend.retrieval.index import construir_indice
from backend.retrieval.tabla import guardar_csv, tabla_recuperacion, titulo_visible

configurar_logs()
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 70, "display.max_rows", 200)

In [ ]:
# --- Ajusta estas constantes ---------------------------------------------------------------------------------------------
ZONA = "test"                  # "test": MOCK y normas públicas (reproducible, versionable) | "val": tus documentos reales (confinados)
NORMA_ARCHIVO  = ""   # vacío = el único PDF de documentos/<ZONA>/normas/; si hay varios, lista y pide nombre o n.º
MANUAL_ARCHIVO = ""   # vacío = el único PDF de documentos/<ZONA>/manuales/; si hay varios, lista y pide nombre o n.º
ARTICULOS = "31-48"            # artículos de la norma a comparar ("31-48", "5,7,10-12"); vacío = todos
COMPARACION = ""               # tu comparación manual (CSV/XLSX en documentos/<ZONA>/comparaciones/); vacío = se omite la validación
COL_ARTICULO, COL_SECCION = "Artículo", "Sección"
TOP_K_MOSTRAR = 3              # candidatos por artículo que se muestran en pantalla (el CSV lleva todos)
MOSTRAR_TEXTO = ZONA == "test" # fragmentos de texto en pantalla: solo en test; en val no se muestra texto de los documentos
# -------------------------------------------------------------------------------------------------------------------------
def resolver(nombre, carpeta, que, defecto=None):
    """PDF a usar: el nombre dado, el n.º de la lista, o (si no se da nada) el único PDF de la carpeta. Si hay varios, lista y se detiene."""
    pdfs = sorted(carpeta.glob("*.pdf"))
    if isinstance(nombre, int):
        elegido = pdfs[nombre - 1] if 1 <= nombre <= len(pdfs) else None
    elif nombre:
        elegido = carpeta / nombre if Path(nombre).name == nombre and (carpeta / nombre).is_file() else None   # solo nombres dentro de la zona
    elif len(pdfs) == 1:
        elegido = pdfs[0]
    else:
        elegido = carpeta / defecto if defecto and (carpeta / defecto).is_file() else None
    if elegido is None:
        print(f"Hay {len(pdfs)} PDF en {carpeta}. Elige la {que} (nombre o n.º) en {que.upper()}_ARCHIVO:")
        for i, f in enumerate(pdfs, 1):
            print(f"   {i}. {f.name}  ({f.stat().st_size / 2**20:.1f} MB){'  [ficticio]' if f.name.startswith('MOCK-DEMO-') else ''}")
        raise SystemExit(f"No pude elegir la {que}: {nombre or '(vacío)'}")
    return elegido

def rango_articulos(texto):
    out = set()
    for parte in filter(None, (p.strip() for p in texto.split(","))):
        a, _, b = parte.partition("-")
        out.update(range(int(a), int(b or a) + 1))
    return out

DEFECTO_TEST = {"norma": "Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf", "manual": "MOCK-DEMO-01.pdf"}   # solo zona test: demo reproducible
d = DEFECTO_TEST if ZONA == "test" else {}
NORMA = resolver(NORMA_ARCHIVO, carpeta_normas(ZONA), "norma", d.get("norma"))
MANUAL = resolver(MANUAL_ARCHIVO, carpeta_manuales(ZONA), "manual", d.get("manual"))
BLOQUES, SALIDA = carpeta_bloques(ZONA), carpeta_salida(ZONA) / "recuperacion"
SALIDA.mkdir(parents=True, exist_ok=True)
carpeta_comparaciones(ZONA).mkdir(parents=True, exist_ok=True)   # aquí va tu comparación manual (CSV/XLSX)
print(f"ZONA={ZONA} · norma: {NORMA.name} · manual: {MANUAL.name}")
print(f"stage={os.environ.get('STAGE')} · embeddings: {s.EMB_PROVIDER} {s.DMR_EMB_MODEL or s.OLLAMA_EMB_MODEL or s.FOUNDRY_AI_EMBED_DEPLOYMENT} · reranker: {s.RERANK_MODEL if s.RERANK_ENABLED else 'apagado'}")
print(f"SUBCHUNK_TOKENS={s.SUBCHUNK_TOKENS} OVERLAP={s.SUBCHUNK_OVERLAP} K_SUBCHUNKS={s.K_SUBCHUNKS} SIM_THRESHOLD={s.SIM_THRESHOLD} MIN_FLOOR={s.MIN_FLOOR} MAX_CANDIDATES={s.MAX_CANDIDATES}")

## 1 · Secciones y chunks (tabla)
Docling real (la primera vez tarda; queda en `output/<zona>/bloques/`). Una fila por sección: ruta, páginas, caracteres, **cuántos sub-chunks** y tokens máximos, si se compara o se excluye (carátula, índice, anexos…) y por qué.

In [ ]:
def cargar(pdf, doc_id, tipo):
    avisos = []
    with progreso(f"Docling · {pdf.name}", "pág") as cb:
        secs = leer_documento(pdf, doc_id, tipo, avisos=avisos, progreso=cb, cache=BLOQUES)
    return secs, tabla_recuperacion(secs, pdf, pdf.stem), avisos

secs_n, t_n, av_n = cargar(NORMA, "N", "normativa")
secs_m, t_m, av_m = cargar(MANUAL, "M", "manual_control")

arts = [x for x in secs_n if x.es_hoja and x.nivel == "articulo"] or [x for x in secs_n if x.es_hoja]
if ARTICULOS:
    quiero = rango_articulos(ARTICULOS)
    arts = [x for x in arts if (m := re.search(r"\d+", x.identificador)) and int(m.group()) in quiero]
incl_m = seleccion.solo_incluidas(secs_m, t_m)
hojas_m = seleccion.ids_seleccionados(t_m)

print(f"Norma: {len(secs_n)} secciones · artículos a comparar: {len(arts)}" + (f" ({ARTICULOS})" if ARTICULOS else ""))
print(f"Manual: {len(secs_m)} secciones · {seleccion.resumen_exclusion(t_m)} Quedan {len(hojas_m)} hojas para comparar.")
for aviso in seleccion.avisos_exclusion(t_m): print(aviso)
inc_n, inc_m = int((t_n["Incierto"] == "⚠").sum()), int((t_m["Incierto"] == "⚠").sum())
print(f"Avisos del seccionado: norma {len(av_n)} · manual {len(av_m)}")
print(f"Secciones con seccionado incierto (⚠): norma {inc_n} de {len(t_n)} · manual {inc_m} de {len(t_m)}. El texto está íntegro; lo dudoso es dónde empieza o termina la sección y su ruta. Revísalas en la tabla (columna Incierto).")
for aviso in [a for a in av_m if a.startswith("Seccionado incierto")][:10]: print("  manual ·", aviso)
assert arts, "No quedó ningún artículo: revisa ARTICULOS y que el PDF sea la norma."
assert hojas_m, "No quedó ninguna hoja del manual para comparar."

In [ ]:
VER = ["Rol", "Nivel", "Identificador", "Título", "Ruta", "Págs", "Caracteres", "Sub-chunks", "Tokens máx.", "Incluir", "Motivo", "Incierto"]
print("MANUAL · resumen de exclusión y de chunks")
display(seleccion.contadores(t_m))
hojas = t_m[t_m["Rol"] != "agrupa"]
print(f"hojas: {len(hojas)} · sub-chunks: {int(hojas['Sub-chunks'].sum())} · secciones con más de 1 sub-chunk: {int((hojas['Sub-chunks'] > 1).sum())} · tokens máx. de un sub-chunk: {int(hojas['Tokens máx.'].max())} (presupuesto SUBCHUNK_TOKENS={s.SUBCHUNK_TOKENS})")
display(t_m[VER].head(60))
art_ids = {x.id for x in arts}
t_arts = t_n[t_n["Id"].isin(art_ids)]
print(f"\nNORMA · {len(t_arts)} artículos a comparar · sub-chunks: {int(t_arts['Sub-chunks'].sum())} · con más de 1 sub-chunk: {int((t_arts['Sub-chunks'] > 1).sum())} · tokens máx.: {int(t_arts['Tokens máx.'].max())}")
display(t_arts[VER].head(40))
guardar_csv(t_m, SALIDA / f"secciones__{MANUAL.stem}.csv"); guardar_csv(t_n, SALIDA / f"secciones__{NORMA.stem[:60]}.csv")

## 2 · Embeddings y FAISS
Embeddings **reales** del stage. Un texto que no cabe en el contexto del modelo se vectoriza por ventanas (multi-vector), sin recortar. Con `RERANK_ENABLED=true` el reranker puntúa el top-`RERANK_TOP_N` de cada vía.

In [ ]:
cliente = ModelClient()
nombres = {"N": NORMA.stem, "M": MANUAL.stem}
with progreso("Embeddings · norma", "sub-chunk") as cb:
    i_n = construir_indice(arts, cliente, nombres, progreso=cb)
with progreso("Embeddings · manual", "sub-chunk") as cb:
    i_m = construir_indice(incl_m, cliente, nombres, progreso=cb)
print(f"norma: {len(i_n)} vectores · manual: {len(i_m)} vectores · dimensión {i_n.vectores.shape[1]}")

## 3 · Recuperación: los pares que recibiría el juez
Dos vías (artículo → manual y manual → artículo), unidas y sin repetir. Se muestran los `TOP_K_MOSTRAR` mejores por artículo; el CSV lleva todos.

### Cómo leer la tabla
Cada fila es un **par (artículo, sección del manual)** que el juez evaluaría una vez. Salen ordenadas por artículo y, dentro de cada uno, por el mejor rango.

| Columna | Qué significa |
|---|---|
| `Artículo`, `Sección`, `Título sección`, `Ruta`, `Págs` | Qué se compara y dónde está la sección en el manual |
| `Origen` | Qué vía propuso el par: `v1` (el artículo la buscó en el manual), `v2` (la sección buscó el artículo en la norma) o `v1+v2` (las dos coinciden: es la señal más fuerte) |
| `Rango v1` | Posición de esa sección entre los candidatos **del artículo** (1 = la más parecida) |
| `Rango v2` | Posición de ese artículo entre los candidatos **de la sección** (1 = el más parecido) |
| `Score v1`, `Score v2` | Similitud coseno (0-1) del mejor sub-chunk de cada lado. Depende del modelo: solo se compara dentro de una misma corrida, no entre modelos |
| `Reranker` | Solo si `RERANK_ENABLED=true`: puntaje 0-1 del reranker; entonces los rangos son los del reranker y el score sigue siendo el coseno |

**Qué significa `—` (en el CSV, celda vacía; antes `NaN`).** No es un error: es "no aplica". Un par que salió solo por una vía no tiene rango ni score en la otra:
- `Rango v2` y `Score v2` en `—` → el artículo apunta a esa sección, pero el artículo no quedó entre los mejores candidatos de la sección (hay otros artículos que se le parecen más). Es habitual con secciones genéricas ("Definiciones", "Generalidades").
- `Rango v1` y `Score v1` en `—` → al revés: la sección apunta al artículo, pero el artículo no la eligió entre sus mejores secciones.
- Con la misma sección y el mismo artículo, el score suele ser el mismo número en ambas vías (la similitud es simétrica); por eso aparece repetido cuando el origen es `v1+v2`.

**Cómo usarla.** Un par que debería cumplirse sale arriba (rangos 1-3) y en `v1+v2`. Si la sección correcta aparece solo en `v2`, o con rango alto en `v1` (por ejemplo 7-10), es un candidato en el límite: se juzga, pero merece mirar si el tope `MAX_CANDIDATES` o el umbral lo dejarían fuera. Los artículos con pocos candidatos o con el score más bajo (celda siguiente) son los que quizá el manual no cubre.


In [ ]:
reranker = textos = None
if s.RERANK_ENABLED:
    from backend.retrieval.reranker import Reranker
    reranker, textos = Reranker(s), {x.id: x.texto_literal for x in [*arts, *incl_m]}
pares, st = construir_pares(i_n, i_m, art_ids, hojas_m, s, reranker, textos)
print(f"pares ingenuos {st['n_pares_ingenuo']} → únicos {st['n_pares_unicos']} (solo vía 1: {st['solo_v1']}, solo vía 2: {st['solo_v2']}, ambas: {st['ambas']})")

por_art = {x.id: x for x in arts}; por_sec = {x.id: x for x in secs_m}
def mejor(p):   # orden de lectura: primero el de mejor rango entre las vías
    return min(r for r in (p.rank_v1, p.rank_v2) if r)
filas = []
for p in pares:
    a, sc = por_art[p.articulo_id], por_sec[p.seccion_id]
    filas.append({"Artículo": a.identificador, "Sección": sc.identificador, "Título sección": titulo_visible(sc, 55), "Ruta": " › ".join(sc.ruta), "Págs": sc.pagina_inicio,
                  "Origen": "+".join(sorted(p.origen)), "Rango v1": p.rank_v1, "Rango v2": p.rank_v2, "Score v1": p.score_v1 and round(p.score_v1, 3),
                  "Score v2": p.score_v2 and round(p.score_v2, 3), "Reranker": p.score_rerank and round(p.score_rerank, 3), "_n": int(re.search(r"\d+", a.identificador).group()) if re.search(r"\d+", a.identificador) else 0, "_m": mejor(p),
                  "_art": p.articulo_id, "_sec": p.seccion_id})
tabla_pares = pd.DataFrame(filas).sort_values(["_n", "_m"])
guardar_csv(tabla_pares.drop(columns=["_n", "_m", "_art", "_sec"]), SALIDA / f"recuperacion__{NORMA.stem[:40]}__{MANUAL.stem}.csv")
vista = tabla_pares.groupby("_art", sort=False).head(TOP_K_MOSTRAR).drop(columns=["_n", "_m", "_art", "_sec"])
vista = vista.dropna(axis=1, how="all")
for col in ("Rango v1", "Rango v2"):
    vista[col] = vista[col].astype("Int64")                      # rangos enteros (no 1.0)
display(vista.astype(object).where(vista.notna(), "—"))      # "—" = no aplica (el par salió solo por la otra vía); ver "Cómo leer la tabla"

In [ ]:
# Diagnósticos
n_cand = tabla_pares.groupby("_art").size()
print(f"candidatos por artículo: mín {n_cand.min()} · mediana {int(n_cand.median())} · máx {n_cand.max()} (piso {s.MIN_FLOOR}, tope {s.MAX_CANDIDATES})")
fuertes = tabla_pares.assign(sc=tabla_pares[["Score v1", "Score v2"]].max(axis=1)).groupby("_art")["sc"].max().sort_values()
print("\nArtículos con el score más bajo (revisa si el manual realmente no los cubre):")
display(pd.DataFrame({"Artículo": [por_art[i].identificador for i in fuertes.index[:8]], "Mejor score": fuertes.values[:8].round(3)}))
huerfanas = hojas_m - set(tabla_pares["_sec"])
print(f"\nSecciones del manual que ningún artículo recupera: {len(huerfanas)} de {len(hojas_m)}")
if huerfanas:
    display(pd.DataFrame([{"Sección": por_sec[i].identificador, "Título": titulo_visible(por_sec[i], 60), "Págs": por_sec[i].pagina_inicio} for i in sorted(huerfanas)][:25]))
if MOSTRAR_TEXTO:   # solo zona test
    print("\nEjemplo (zona test): el mejor candidato de los primeros artículos")
    for a_id in list(tabla_pares["_art"].drop_duplicates())[:3]:
        fila = tabla_pares[tabla_pares["_art"] == a_id].iloc[0]
        print(f"\n— {por_art[a_id].identificador}: {por_art[a_id].texto_literal[:260]!r}\n  → {fila['Sección']}: {por_sec[fila['_sec']].texto_literal[:260]!r}")

## 4 · Validación contra tu comparación manual (opcional)
Si pones `COMPARACION`, mide para cada par (artículo, sección) de tu archivo si quedó entre los candidatos y su rango en cada vía. Solo se usan identificadores. Una fila por artículo; sección vacía = omisión; varias secciones separadas por `;`. En Excel, da formato texto a la columna de secciones (`4.10` se guardaría como `4,1`).

In [ ]:
if not COMPARACION:
    print("Sin COMPARACION: se omite la validación.")
else:
    from backend.retrieval.validacion import evaluar_comparacion, leer_comparacion
    from scripts.medir_recuperacion import fmt
    archivo = Path(COMPARACION) if Path(COMPARACION).is_absolute() else carpeta_comparaciones(ZONA) / COMPARACION
    filas_cm, sin_numero = leer_comparacion(archivo, COL_ARTICULO, COL_SECCION)
    num = lambda x: int(re.search(r"\d+", x.identificador).group())
    ev = evaluar_comparacion(filas_cm, {num(x): x.id for x in arts}, incl_m, i_n, i_m, pares)
    fe = ev["filas"]; n = len(fe)
    print(f"{len(filas_cm)} filas ({sin_numero} sin número de artículo) · evaluadas {n} · omisiones {len(ev['omisiones'])}")
    if ev["sin_articulo"]: print("Artículos de tu archivo fuera de los seleccionados:", sorted(set(ev["sin_articulo"])))
    if ev["sin_seccion"]: print("Secciones que no encuentro entre las hojas incluidas (excluida, otro id o seccionado distinto):", ev["sin_seccion"])
    if n:
        print(f"recall@candidatos: {fmt(sum(f['en_candidatos'] for f in fe), n)}")
        for via in ("rango_v1", "rango_v2"):
            r = [f[via] for f in fe]
            print(f"  {via[-2:]}: " + "  ".join(f"k={k}: {100 * sum(x is not None and x <= k for x in r) / n:.0f} %" for k in (1, 3, 5, 10)) + f"  MRR={sum(1 / x for x in r if x) / n:.3f}")
        print("Fallos (artículo, sección, rango v1, rango v2):", [(f["articulo"], f["seccion"], f["rango_v1"], f["rango_v2"]) for f in fe if not f["en_candidatos"]])
        pd.DataFrame(fe).to_csv(SALIDA / f"validacion__{archivo.stem}.csv", index=False)

## 5 · Qué se guardó y qué se puede subir
Todo en `output/<ZONA>/recuperacion/`: tablas de secciones, pares recuperados y la validación. **`test`** se puede versionar (CSV con ids, títulos y scores de MOCK y normas públicas). **`val`** nunca (ignorado por git). Para comparar modelos de embeddings y rerankers sobre estos mismos documentos, usa `03c_comparacion_embeddings_reranker.ipynb`.